In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms.v2 as transforms
import timm

# Implementation of the core Additive Angular Margin Loss (ArcFace Module)
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        # Initialize class center weights layer
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        # 1. Cosine similarity calculation: L2 normalized weights and inputs
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        
        # 2. Apply trigonometric identity for cos(theta + m)
        phi = cosine * self.cos_m - sine * self.sin_m
        # Handle boundary conditions where theta + m > pi
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        # 3. Transform targets using one-hot mapping masks
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

In [ ]:
DATA_DIR = "./datasets/Retail-YU_reformed"
BATCH_SIZE = 64

# Basic normalization metrics inheriting ImageNet defaults
data_transforms = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=f"{DATA_DIR}/train", transform=data_transforms)
val_dataset = ImageFolder(root=f"{DATA_DIR}/val", transform=data_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

NUM_CLASSES = len(train_dataset.classes)
print(f"Dataset configuration finalized. Tracked Classes Count: {NUM_CLASSES}")

In [ ]:
import lightning as L
from lightning.pytorch.tuner import Tuner
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# 1. Bring over the ArcMarginProduct layer from your previous setup
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

# 2. Define the complete Lightning System
class ArcFaceVendingSystem(L.LightningModule):
    def __init__(self, num_classes, embedding_size=512, lr=1e-4):
        super().__init__()
        # Essential statement to log hyperparameter changes (required for LR Finder)
        self.save_hyperparameters()
        self.lr = lr
        
        # Identity-overwritten backbone initialization
        self.backbone = timm.create_model('hf_hub:timm/convnext_tiny.fb_in22k_ft_in1k', pretrained=True)
        num_features = self.backbone.num_features
        self.backbone.head.fc = nn.Identity()
        
        self.bn_layer = nn.BatchNorm1d(num_features)
        self.fc_embedding = nn.Linear(num_features, embedding_size)
        self.arcface_head = ArcMarginProduct(in_features=embedding_size, out_features=num_classes)
        self.criterion = nn.CrossEntropyLoss()

    def extract_embeddings(self, x):
        features = self.backbone(x)
        features = self.bn_layer(features)
        embeddings = self.fc_embedding(features)
        return F.normalize(embeddings)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        features = self.backbone(images)
        features = self.bn_layer(features)
        embeddings = self.fc_embedding(features)
        
        logits = self.arcface_head(embeddings, labels)
        loss = self.criterion(logits, labels)
        
        # Calculate metric monitoring
        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        
        # Log values to show them automatically inside the progress bar
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        features = self.backbone(images)
        features = self.bn_layer(features)
        embeddings = self.fc_embedding(features)
        
        logits = self.arcface_head(embeddings, labels)
        loss = self.criterion(logits, labels)
        
        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        # Always reference self.lr so that the Tuner can rewrite it dynamically
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Create instance targeting your class count variable
model_system = ArcFaceVendingSystem(num_classes=NUM_CLASSES)

# Instantiate basic multi-hardware execution profile runner
trainer = L.Trainer(
    max_epochs=10,
    accelerator="auto", # Automatically detects and targets CUDA if available
    devices=1,
    enable_progress_bar=True
)

# Initialize the automated Tuning engine
tuner = Tuner(trainer)

torch.set_float32_matmul_precision('medium')

print("Starting automatic learning rate range test exploration...")
lr_finder = tuner.lr_find(model_system, train_dataloaders=train_loader)

# Pick out the suggested optimization value automatically 
new_lr = lr_finder.suggestion()
print(f"Optimal inflection point identified! Setting learning rate to: {new_lr:.6f}")

# Update the model system to apply the newly discovered rate configuration
model_system.lr = new_lr

# Optional: Render the traditional LR Loss-Slope breakdown graph
# fig = lr_finder.plot(suggest=True)
# fig.show()

In [ ]:
trainer.fit(model_system, train_dataloaders=train_loader, val_dataloaders=val_loader)

In [ ]:
#save model
import torch
# Extract the pure PyTorch state dictionary from the Lightning system
torch.save(model_system.state_dict(), "./models/tuned/items_classification_convnext_tiny.fb_in22k_ft_in1k.pt")
print("Pure PyTorch weights saved cleanly.")

In [ ]:
import os
import numpy as np
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 1. Double check and dynamically filter out any empty folders right before loading
gallery_dir = f"{DATA_DIR}/gallery"
for folder in os.listdir(gallery_dir):
    f_path = os.path.join(gallery_dir, folder)
    if os.path.isdir(f_path) and not os.listdir(f_path):
        os.rmdir(f_path) # Clear it out so ImageFolder doesn't throw a fit

# 2. Load your custom gallery dataset safely
gallery_dataset = ImageFolder(root=gallery_dir, transform=data_transforms)
gallery_loader = DataLoader(gallery_dataset, batch_size=1, shuffle=False)

# 3. Put the Lightning system into evaluation mode
model_system.eval()
current_device = model_system.device
gallery_database = {}

print(f"Generating target embedding registry using device: {current_device}...")

with torch.no_grad():
    for img, label_idx in gallery_loader:
        img = img.to(current_device)
        
        # Extract the normalized 512-dim embedding vector
        embedding = model_system.extract_embeddings(img).cpu().numpy().flatten()
        
        class_name = gallery_dataset.classes[label_idx]
        
        if class_name not in gallery_database:
            gallery_database[class_name] = []
        gallery_database[class_name].append(embedding)

# 4. Average out embeddings if a class has multiple references
for class_name in gallery_database:
    gallery_database[class_name] = np.mean(gallery_database[class_name], axis=0)

# 5. Save your reference dictionary to disk
os.makedirs("./models/tuned", exist_ok=True)
np.save("./models/tuned/items_classification.npy", gallery_database, allow_pickle=True)
print(f"Successfully registered {len(gallery_database)} unique product reference keys.")

In [ ]:
def classify_product_crop(image_path, model, reference_gallery, transform):
    # 1. Load and prep the image (assumes it was already square-padded via split script)
    from PIL import Image
    img = Image.open(image_path)
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # 2. Extract embedding
    model.eval()
    with torch.no_grad():
        query_embedding = model.extract_embeddings(img_tensor).cpu().numpy().flatten()
    
    # 3. Compute cosine similarity against all gallery items
    best_match = None
    highest_score = -1.0
    
    for class_name, ref_embedding in reference_gallery.items():
        # Cosine similarity formula for normalized vectors: dot product
        similarity = np.dot(query_embedding, ref_embedding)
        if similarity > highest_score:
            highest_score = similarity
            best_match = class_name
            
    return best_match, highest_score

# Usage example:
# match, score = classify_product_crop("path_to_vending_crop.jpg", model, gallery_database, data_transforms)
# print(f"Identified product: {match} with confidence score: {score:.4f}")